In [1]:
from pathlib import Path
from data import load_molecules_split, MoleculeData, BONDS
import os
import sys
from tqdm import tqdm

In [2]:
N_MAX = 29
# E_MAX = 56
E_MAX = N_MAX * (N_MAX - 1)

In [3]:
absolute_path = Path(os.getcwd()).resolve()
load_dir = absolute_path / "preprocessed_data"
train_data = load_molecules_split(load_dir / 'train', num_mols=20)

Loading split=train:   0%|          | 0/20 [00:00<?, ?it/s]

{'file': 'mol_000000.npz', 'smiles': 'NC1(CC(=O)O)COC1', 'molblock': '\n     RDKit          3D\n\n 18 18  0  0  0  0  0  0  0  0999 V2000\n    0.6647    1.7207    0.4392 N   0  0  0  0  0  0  0  0  0  0  0  0\n    1.4641    2.1890    0.0269 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.1752    2.0644   -0.0157 H   0  0  0  0  0  0  0  0  0  0  0  0\n    0.7604    0.2960    0.2675 C   0  0  1  0  0  0  0  0  0  0  0  0\n   -0.4960   -0.3675    0.8310 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.3741   -1.4486    0.9006 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.6780    0.0227    1.8356 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.7202   -0.0781   -0.0010 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -1.8447    0.8171   -0.7941 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.6982   -0.9534    0.2493 O   0  0  0  0  0  0  0  0  0  0  0  0\n   -3.4880   -0.7455   -0.2755 H   0  0  0  0  0  0  0  0  0  0  0  0\n    1.2055   -0.2722   -1.1069 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.545

Loading split=train: 100%|██████████| 20/20 [00:03<00:00,  6.57it/s]

{'file': 'mol_000001.npz', 'smiles': 'NC1(CC(=O)O)COC1', 'molblock': '\n     RDKit          3D\n\n 18 18  0  0  0  0  0  0  0  0999 V2000\n   -0.5904   -1.7435    0.3712 N   0  0  0  0  0  0  0  0  0  0  0  0\n    0.2963   -1.8638    0.8519 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.6182   -2.4078   -0.3952 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -0.7228   -0.4004   -0.1176 C   0  0  1  0  0  0  0  0  0  0  0  0\n    0.4569    0.0849   -1.0006 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.5774   -0.6121   -1.8326 H   0  0  0  0  0  0  0  0  0  0  0  0\n    0.2602    1.0870   -1.3781 H   0  0  0  0  0  0  0  0  0  0  0  0\n    1.7128    0.0959   -0.1748 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.2383   -0.8789    0.2988 O   0  0  0  0  0  0  0  0  0  0  0  0\n    2.1750    1.3324    0.0299 O   0  0  0  0  0  0  0  0  0  0  0  0\n    2.9688    1.3166    0.5886 H   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.1039   -0.0930   -0.7249 C   0  0  0  0  0  0  0  0  0  0  0  0\n   -2.119

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Geometry import Point3D
import py3Dmol
import numpy as np
import jax.numpy as jnp

In [5]:
def viz_molblock(molblock: str, width: int = 400, height: int = 400):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(molblock, "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.show()

def viz_from_molblock_and_coords(
    molblock: str,
    coords,  # (N, 3) JAX/NumPy array
    width: int = 400,
    height: int = 400,
):

    # Reconstruct the exact molecule template (preserve explicit hydrogens)
    mol = Chem.MolFromMolBlock(molblock, removeHs=False, sanitize=True)
    if mol is None:
        raise ValueError("MolFromMolBlock failed. MolBlock may be invalid or incompatible.")

    coords = np.asarray(coords, dtype=np.float64)
    n = mol.GetNumAtoms()
    if coords.shape != (n, 3):
        raise ValueError(f"coords shape {coords.shape} does not match mol atoms {(n, 3)}.")

    # Overwrite conformers with your coordinates
    conf = Chem.Conformer(n)
    for i in range(n):
        x, y, z = coords[i]
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

    mol.RemoveAllConformers()
    mol.AddConformer(conf, assignId=True)

    # Convert back to MolBlock for py3Dmol rendering
    new_block = Chem.MolToMolBlock(mol, confId=0)
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(new_block, "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.show()

In [16]:
import random

rand_idx = random.randint(0, len(train_data) - 1)
sample_mol, smiles, molblock = train_data[rand_idx]
print(smiles)

CCOC1=NCCOC1


In [17]:
viz_molblock(molblock)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [18]:
for idx in range(len(train_data)):

    sample_mol, smiles, molblock = train_data[idx]
    print(smiles)
    viz_molblock(molblock)

NC1(CC(=O)O)COC1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

NC1(CC(=O)O)COC1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

NC1(CC(=O)O)COC1


...

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

N#CCO[C@@H]1C=CCC1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

N#CCO[C@@H]1C=CCC1


3Dmol.js failed to load for some reason. Please check your browser console for error messages.